# LC #424 — Longest Repeating Character Replacement
**Category:** Sliding Window + HashMap | **Difficulty:** Medium | **Day 2**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>The Problem:</strong> Given string <code>s</code> and integer <code>k</code>,
return the length of the longest substring containing the same letter after at most <code>k</code>
character replacements.
</div>

**Examples:**
```
"AABABBA", k=1 → 4  ("AABA" or "ABBB": replace one B → AAAA, length 4)
"ABAB",    k=2 → 4  (replace both Bs → "AAAA")
"AAAA",    k=0 → 4
```

**Core Insight:** A window is valid if `(window_size - max_count_in_window) ≤ k`. The left term counts how many characters would need to be replaced. If that's ≤ k, we can make the whole window uniform.

## Mental Models

**1. The Validity Formula**
`replacements_needed = window_size - count_of_most_frequent_char`
If `replacements_needed ≤ k`, the window is valid. This is the invariant.

**2. Why We Don't Update max_count on Shrink**
We only want the maximum valid window ever seen. `max_count` never decreases. Even if the most frequent char leaves the window, we don't shrink `max_count` — this can only prevent us from growing the window, not from finding a larger valid window later.

**3. The Sliding Window Contract**
Right pointer expands. When the window becomes invalid, left pointer moves up by exactly 1 (not until valid again). This maintains O(n) total movement.

In [ ]:
# Brute Force — O(n²) per window, O(1) per check
from collections import Counter

def characterReplacement_brute(s: str, k: int) -> int:
    max_len = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            window = s[i:j]
            most_common_count = Counter(window).most_common(1)[0][1]
            if len(window) - most_common_count <= k:
                max_len = max(max_len, len(window))
    return max_len

print(characterReplacement_brute("AABABBA", 1))  # 4
print(characterReplacement_brute("ABAB", 2))     # 4

## Why We Never Decrease max_count

Consider `"AABABBA"`, k=1:

Key insight: `max_count` tracks the best frequency we've seen in *any* window state.
- When we shrink left, max_count doesn't decrease.
- This means the window can only *stay the same size or grow* — we're searching for the maximum.
- If the actual max_count in the current window is less than stored max_count, the window is "artificially" large — but we just move left by 1, maintaining the search.

This is a valid optimization because we're looking for the largest window, not validating all windows.

In [ ]:
# Optimal — O(n) time, O(1) space (at most 26 distinct chars)

def characterReplacement(s: str, k: int) -> int:
    count = {}
    left = 0
    max_count = 0   # max frequency of any single char in current window
    result = 0

    for right in range(len(s)):
        count[s[right]] = count.get(s[right], 0) + 1
        max_count = max(max_count, count[s[right]])

        # Characters to replace = window_size - most_frequent_count
        # If > k, shrink by 1 from left
        if (right - left + 1) - max_count > k:
            count[s[left]] -= 1
            left += 1

        result = max(result, right - left + 1)

    return result

# Test
print(characterReplacement("AABABBA", 1))   # 4
print(characterReplacement("ABAB", 2))       # 4
print(characterReplacement("AAAA", 0))       # 4
print(characterReplacement("A", 0))          # 1

## Complexity Analysis

| Approach | Time | Space |
|----------|------|-------|
| Brute Force | O(n² × 26) | O(26) |
| **Sliding Window** | **O(n)** | **O(26) = O(1)** |

**Why O(1) space?** `count` can have at most 26 keys (uppercase English letters). That's a fixed-size map.

**Why O(n) time?** Left and right each move at most n steps total. The inner operations (update count, max_count) are O(1). Total: O(n).

## Interview Q&A

**Q: Why don't we decrement max_count when shrinking the window?**
A: We're looking for the *maximum length*, so max_count should reflect the best window we've ever seen. If we shrink past the max-count character, the window is now "conservatively" sized — we'll grow it again when we find a better character. Keeping max_count high prevents us from mistakenly invalidating a window that's the same size as our target.

**Q: Is the window always valid?**
A: No — the window may temporarily be invalid (when we shrink left, we don't shrink until valid). The key insight: we're searching for the maximum window, so we only grow and move right by 1. We never shrink below the current max.

**Q: What changes if characters are lowercase or Unicode?**
A: The count map grows, but the algorithm is identical. Space would be O(charset_size) — still effectively O(1) for bounded alphabets.

**Q: Time-series analogy?**
A: "Longest period where a metric stayed within k standard deviations of its dominant value" — same invariant: (window_size - count_of_dominant_state) ≤ k.

## The Citi Angle

**The "dominant state window" pattern:** How long did a server remain in a predominantly "healthy" state, tolerating up to k anomalous readings?

```python
# Longest stretch where srv-01 was "healthy" with at most 1 spike
states = ["H","H","S","H","H","S","S","H","H","H"]  # H=healthy, S=spike
k = 1

count = {}
left = 0
max_count = 0
longest_healthy = 0

for right, state in enumerate(states):
    count[state] = count.get(state, 0) + 1
    max_count = max(max_count, count[state])
    if (right - left + 1) - max_count > k:
        count[states[left]] -= 1
        left += 1
    longest_healthy = max(longest_healthy, right - left + 1)

print(f"Longest predominantly-healthy window: {longest_healthy} readings")
```

**Interview tie-in:** "Character replacement maps directly to alert tolerance windows — the longest period a server was in a dominant state, tolerating at most k deviations. That's a real capacity planning metric."

## Summary

| | Value |
|--|--|
| **Pattern** | Sliding Window + frequency map |
| **Time** | O(n) |
| **Space** | O(26) = O(1) |
| **Invariant** | `(right - left + 1) - max_count ≤ k` |
| **Say in interview** | "Window valid if (size - max_freq) ≤ k. Expand right, shrink left by 1 when invalid. max_count never decreases." |